In [ ]:
import os
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset
from trl import DPOTrainer, DPOConfig
from peft import LoraConfig, get_peft_model
from sklearn.model_selection import train_test_split
import csv
from tqdm import tqdm

# Set Hugging Face access token
os.environ["HF_TOKEN"] = "REDACTED"
# WARNING: Never share your access token publicly or commit it to version control!

torch.amp.autocast_mode_dict = {'cuda': torch.cuda.amp.autocast}
model_name = "google/gemma-3-1b-it"  # Update for Gemma-2 2B model
print("Loading model and tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    clean_up_tokenization_spaces=True,
    token=os.environ["HF_TOKEN"]
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def get_max_length(dataset, tokenizer, key='prompt'):
    max_length = 0
    for item in dataset[key]:
        length = len(tokenizer.encode(item))
        max_length = max(max_length, length)
    return max_length

df = pd.read_csv("reformatted_prompts.csv")
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

def clean_text(text):
    if pd.isna(text) or not isinstance(text, str):
        return ""
    return text.strip()

def create_dict_from_dataframe(df):
    return {
        'prompt': [clean_text(x) for x in df['input'].tolist()],
        'chosen': [clean_text(x) for x in df['accepted'].tolist()],
        'rejected': [clean_text(x) for x in df['rejected'].tolist()]
    }

train_dataset = Dataset.from_dict(create_dict_from_dataframe(train_df))
valid_dataset = Dataset.from_dict(create_dict_from_dataframe(test_df))

# Calculate max lengths with some safety margin
max_prompt_length = min(get_max_length(train_dataset, tokenizer, 'prompt'), 2048)
max_chosen_length = min(get_max_length(train_dataset, tokenizer, 'chosen'), 2048)
max_rejected_length = min(get_max_length(train_dataset, tokenizer, 'rejected'), 2048)

max_length = min(max(max_prompt_length, max_chosen_length, max_rejected_length) + 8, 2048)

print(f"Calculated max length: {max_length}")

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    token=os.environ["HF_TOKEN"]
)

model_ref = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    token=os.environ["HF_TOKEN"]
)

model.config.max_length = max_length
model_ref.config.max_length = max_length

peft_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=['gate_proj', 'down_proj', 'up_proj', 'q_proj', 'v_proj', 'k_proj', 'o_proj']
)

model = get_peft_model(model, peft_config)
model_ref = get_peft_model(model_ref, peft_config)

dpo_config = DPOConfig(
    output_dir='./gemma2-2b-dpo-results',
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=64,
    remove_unused_columns=False,
    learning_rate=1e-5,
    max_steps=400,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    warmup_steps=100,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=max_length,
    max_prompt_length=max_prompt_length,
    max_completion_length=max(max_chosen_length, max_rejected_length),
    beta=0.1,
    weight_decay=0.05,
)

def preprocess_function(examples):
    for key in ['prompt', 'chosen', 'rejected']:
        if key in examples:
            examples[key] = [str(text) for text in examples[key]]
    return examples

train_dataset = train_dataset.map(preprocess_function, batched=True)
valid_dataset = valid_dataset.map(preprocess_function, batched=True)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=model_ref,
    args=dpo_config,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer
)


def generate_responses(model, dataset, tokenizer, max_length, filename, num_datapoints=200):
    responses = []
    
    limited_dataset = dataset.select(range(min(num_datapoints, len(dataset))))
    
    for item in tqdm(limited_dataset, desc=f"Generating responses for {filename}", total=num_datapoints):
        inputs = tokenizer(item['prompt'], return_tensors='pt', padding=True, truncation=True, max_length=max_length).to(model.device)
        outputs = model.generate(
            **inputs,
            max_length=len(inputs["input_ids"][0]) + 200,
            num_beams=4,
            length_penalty=2.0,
            early_stopping=False,
            no_repeat_ngram_size=3,
            do_sample=False,
            temperature=0.000001
        )
        model_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        responses.append({
            'prompt': item['prompt'],
            'model_response': model_response,
            'accepted_response': item['chosen']
        })

    with open(filename, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=['prompt', 'model_response', 'accepted_response'])
        writer.writeheader()
        writer.writerows(responses)

    print(f"Responses saved to {filename}")

print("Generating responses for the first 500 datapoints before training...")
generate_responses(model, valid_dataset, tokenizer, max_length, "responses_before_training.csv", num_datapoints=20)

print("Starting DPO training...")
dpo_trainer.train()
dpo_trainer.model.save_pretrained("final_checkpoint")
tokenizer.save_pretrained("final_checkpoint")
print("Training completed successfully")

model.save_pretrained('./gemma2-2b-dpo-results')
tokenizer.save_pretrained('./gemma2-2b-dpo-results')

print("Generating responses for the first 500 datapoints after training...")
generate_responses(model, valid_dataset, tokenizer, max_length, "responses_after_training.csv", num_datapoints=20)


Loading model and tokenizer...
Calculated max length: 482


Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

Extracting prompt in train dataset:   0%|          | 0/160 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/160 [00:00<?, ? examples/s]

...

It is strongly recommended to train Gemma3 models with the `eager` attention implementation instead of `sdpa`. Use `eager` with `AutoModelForCausalLM.from_pretrained('<path-to-checkpoint>', attn_implementation='eager')`.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
50,0.217200,0.271960,1.345190,0.039533,1.000000,1.305657,-229.051102,-171.120255,-5.395008,-5.343261
100,0.003100,0.025991,1.389967,-4.936252,1.000000,6.326218,-228.603317,-220.878098,-5.517996,-5.462707
150,0.000300,0.016927,-0.248163,-9.117772,1.000000,8.869608,-244.984619,-262.693298,-5.846123,-5.859618
200,0.000200,0.015047,-1.084291,-11.014959,1.000000,9.930669,-253.345901,-281.665192,-5.999282,-6.049417
250,0.000100,0.013649,-1.628608,-12.259280,1.000000,10.630672,-258.789093,-294.108398,-6.098183,-6.173076
300,0.000100,0.012595,-2.006216,-13.102480,1.000000,11.096265,-262.565125,-302.540375,-6.163133,-6.254478
350,0.000100,0.012172,-2.232808,-13.603256,1.000000,11.370448,-264.831055,-307.548157,-6.201273,-6.302423
400,0.000100,0.011981,-2.317053,-13.790854,1.000000,11.473802,-265.673523,-309.424103,-6.214833,-6.319394


Training completed successfully
Generating responses for the first 500 datapoints after training...


Generating responses for responses_after_training.csv:   0%|          | 0/20 [00:00<?, ?it/s]/opt/conda/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-06` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `64` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
Gener

Responses saved to responses_after_training.csv


In [ ]:
torch.cuda.empty_cache()